# 19. DBSCAN
**Use case:** Insurance — Catastrophe claims hotspots after a hailstorm or tornado

After a severe storm, thousands of first-notice-of-loss reports arrive within days, and the carrier must decide where to send a limited number of field adjusters. Policyholders in the hardest-hit areas suffer if adjusters are spread thinly, while isolated claims far from the storm path may be unrelated damage or occasionally suspicious. Each claim has a geocoded address, claim amount, policy age and reporting date. Success means hotspots that follow the actual storm path, a deployment plan sized by claims and dollars, and a short, prioritised review queue for isolated claims - without treating any claim unfairly.

> 💡 Picture a beach from above: groups of people huddled under umbrellas form clusters, while a lone jogger is nobody's neighbour. DBSCAN says: if enough people are within arm's reach of you, you are in a crowd; crowds that touch merge; anyone alone is just a passer-by.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: Spot 2 hotspots and 2 loners among 11 claims
Apply DBSCAN's core, border and noise rules by hand to 11 claim locations with eps = 1 km and min_pts = 3.

| 5W + H | |
|---|---|
| **Who** | A catastrophe claims manager learning how hotspots are found automatically. |
| **What** | Group 11 hail claims (x, y in km) into dense hotspots and flag isolated claims. |
| **When** | The morning after a hailstorm, when the first claims arrive. |
| **Where** | A pure numpy script with the points typed in. |
| **Why** | Adjuster teams should go where claims are dense; isolated claims need a different, desk-review path. |
| **How** | Compute all pairwise distances, count neighbours within eps, mark core points, expand clusters from cores with a queue, and label the rest as noise. |

**🧠 ML concepts:** Density-based clustering, eps neighbourhood, min_samples / min_pts, Core, border and noise points, No need to choose the number of clusters  
**🐍 Python concepts:** Dict of tuples, numpy broadcasting for pairwise distances, Dict comprehension with nested list comprehension, List used as a queue with pop and extend, continue in loops, set() of labels

In [ ]:
# Easy: find hail-claim hotspots by hand with DBSCAN's two rules (eps, min_pts)
import numpy as np

# 🐍 PY: a dict mapping claim id -> (x_km, y_km) position east/north of the city centre
claims = {
    "C1": (1.0, 1.0), "C2": (1.5, 1.2), "C3": (1.2, 0.6), "C4": (0.8, 1.4),   # neighbourhood A
    "C5": (6.0, 6.0), "C6": (6.4, 6.3), "C7": (5.7, 6.5), "C8": (6.2, 5.6),   # neighbourhood B
    "C9": (3.5, 9.0),                                                          # lone claim far away
    "C10": (9.5, 1.0),                                                         # another lone claim
    "C11": (2.2, 1.2),                                                         # edge of A
}
EPS = 1.0      # km - "close enough to be the same hotspot"
MIN_PTS = 3    # a point plus at least 2 neighbours makes a dense core

# 🐍 PY: list() of dict keys gives a fixed order; np.array of values gives an N x 2 matrix
ids = list(claims)
pts = np.array(list(claims.values()))

# 🧠 ML: step 1 - for each claim, count how many claims (itself included) lie within eps
# 🐍 PY: broadcasting pts[:, None] - pts[None, :] gives every pairwise difference at once
dist = np.sqrt(((pts[:, None, :] - pts[None, :, :]) ** 2).sum(axis=2))
neighbours = {ids[i]: [ids[j] for j in range(len(ids)) if dist[i, j] <= EPS] for i in range(len(ids))}
for cid, nb in neighbours.items():
    # 🧠 ML: a CORE point has >= MIN_PTS points inside its eps circle
    kind = "CORE" if len(nb) >= MIN_PTS else "not core (border if a core point reaches it)"
    print(f"{cid:<4} neighbours={nb}  -> {kind}")

# 🧠 ML: step 2 - grow clusters from core points; anything reachable joins the cluster
label = {cid: None for cid in ids}
cluster_id = 0
for cid in ids:
    if label[cid] is not None or len(neighbours[cid]) < MIN_PTS:
        continue
    cluster_id += 1
    # 🐍 PY: a list used as a queue - pop() takes the next point to expand
    queue = [cid]
    while queue:
        p = queue.pop()
        if label[p] is not None:
            continue
        label[p] = cluster_id
        if len(neighbours[p]) >= MIN_PTS:
            queue.extend(neighbours[p])

# 🧠 ML: step 3 - anything never reached is NOISE (label -1): flag for individual review
# 🐍 PY: dict comprehension with a conditional value
label = {c: (l if l is not None else -1) for c, l in label.items()}
print("\nFinal labels:", label)
for k in sorted(set(label.values())):
    members = [c for c, l in label.items() if l == k]
    name = "NOISE - review individually" if k == -1 else f"Hotspot {k} - send an adjuster team"
    print(f"  {name}: {members}")

**Try this next:**
- Set EPS = 0.5 and see the hotspots break up.
- Set MIN_PTS = 5 and see which hotspot survives.
- Move C9 to (2.0, 2.0) - does it join hotspot 1 as a border point?

## MEDIUM: sklearn DBSCAN on 1,010 hailstorm claims
Cluster synthetic claim latitude/longitude with scikit-learn's DBSCAN after converting to km, compare eps values, and summarise hotspots by count and dollars.

| 5W + H | |
|---|---|
| **Who** | A property insurer's catastrophe response team. |
| **What** | Find hail-swath hotspots among about 1,000 claims and size each by total dollars. |
| **When** | In the first 48 hours after a hailstorm, as first notice of loss reports pour in. |
| **Where** | A Python job on the claims data warehouse, output shared with the field operations lead. |
| **Why** | Hail falls in long thin swaths, not round blobs; DBSCAN follows any shape and does not need to know how many swaths there are. |
| **How** | Generate three swaths plus scattered claims, project lat/lon to km, test eps values, fit DBSCAN with eps 0.6 km, compute silhouette, summarise by groupby and compare with the known swaths. |

**🧠 ML concepts:** DBSCAN in scikit-learn, Choosing eps in physical units, Effect of eps on clusters and noise, Silhouette score excluding noise, Comparing clusters with known groups  
**🐍 Python concepts:** List of dicts as configuration, Tuple unpacking of nested tuples, np.column_stack, Set difference to drop the noise label, groupby with named aggregation, pd.crosstab

In [ ]:
# Medium: sklearn DBSCAN on synthetic hailstorm claim locations around a metro area
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

np.random.seed(42)
# Hail swaths are long and thin (storm tracks), not round - a shape DBSCAN handles well
# 🐍 PY: a list of dicts describes each synthetic hail swath
swaths = [
    {"name": "North swath", "start": (36.20, -97.15), "end": (36.24, -97.00), "n": 450},
    {"name": "East swath",  "start": (36.08, -96.98), "end": (36.15, -96.92), "n": 300},
    {"name": "Town core",   "start": (36.11, -97.07), "end": (36.12, -97.06), "n": 200},
]
frames = []
for sw in swaths:
    t = np.random.rand(sw["n"])
    # 🐍 PY: tuple unpacking pulls lat/lon out of the start and end tuples
    (lat0, lon0), (lat1, lon1) = sw["start"], sw["end"]
    lat = lat0 + t * (lat1 - lat0) + np.random.normal(0, 0.004, sw["n"])
    lon = lon0 + t * (lon1 - lon0) + np.random.normal(0, 0.004, sw["n"])
    frames.append(pd.DataFrame({"lat": lat, "lon": lon, "true_area": sw["name"]}))
# Scattered claims: wind-only damage, late reports, or possibly suspicious
n_scatter = 60
frames.append(pd.DataFrame({"lat": np.random.uniform(36.00, 36.30, n_scatter),
                            "lon": np.random.uniform(-97.20, -96.85, n_scatter),
                            "true_area": "Scattered"}))
claims = pd.concat(frames, ignore_index=True)
claims["claim_amount"] = np.random.lognormal(mean=9.3, sigma=0.5, size=len(claims)).round(0)
print(f"{len(claims)} claims loaded. Sample:\n{claims.head(3).to_string()}\n")

# 🧠 ML: convert degrees to km so eps means something physical (1 deg lat ~ 111 km)
lat_ref = np.radians(claims.lat.mean())
X = np.column_stack([claims.lat * 111.0, claims.lon * 111.0 * np.cos(lat_ref)])

# 🧠 ML: eps = neighbourhood radius (km), min_samples = claims needed to call a spot dense
print(" eps_km | clusters | noise claims")
for eps in [0.3, 0.6, 1.0, 2.0]:
    lab = DBSCAN(eps=eps, min_samples=10).fit_predict(X)
    # 🐍 PY: set() removes duplicates; subtract the noise label -1 to count real clusters
    n_clusters = len(set(lab) - {-1})
    print(f"  {eps:>4} |    {n_clusters:>2}    |   {np.sum(lab == -1):>4}")

model = DBSCAN(eps=0.6, min_samples=10)
claims["cluster"] = model.fit_predict(X)
# 🐍 PY: a boolean mask keeps only rows where the condition is True
mask = claims.cluster != -1
# 🧠 ML: silhouette (-1 to 1) measures how well separated clusters are; computed without noise points
print(f"\nChosen eps=0.6 km. Silhouette (non-noise) = {silhouette_score(X[mask], claims.cluster[mask]):.3f}")

# 🐍 PY: groupby + agg with named outputs builds a per-hotspot summary table
summary = (claims[mask].groupby("cluster")
           .agg(claims=("claim_amount", "size"), total_usd=("claim_amount", "sum"),
                centre_lat=("lat", "mean"), centre_lon=("lon", "mean"))
           .sort_values("total_usd", ascending=False))
summary["total_usd"] = summary.total_usd.map("${:,.0f}".format)
print("\nHotspots ranked by exposure:")
print(summary.round(4).to_string())

# 🐍 PY: pd.crosstab counts how true areas map onto found clusters
print("\nTrue area vs DBSCAN label:")
print(pd.crosstab(claims.true_area, claims.cluster).to_string())

# Business interpretation
noise = claims[claims.cluster == -1]
print(f"\n{mask.sum()} claims sit in {claims.cluster[mask].nunique()} hotspots -> deploy adjuster teams to each centre.")
print(f"{len(noise)} isolated claims (${noise.claim_amount.sum():,.0f}) -> desk review; "
      "a hail claim far from any swath deserves a closer look.")

**Try this next:**
- Change min_samples from 10 to 30 and watch the smallest swath behave.
- Skip the km conversion and cluster raw degrees with eps=0.006 - is the result the same?
- Add a fourth swath with only 15 claims and see whether it is found or treated as noise.

## HARD: Haversine DBSCAN hotspot engine with k-distance eps and noise triage
Cluster claims on true great-circle distance, pick eps from the k-distance knee, compare with k-Means, prioritise noise claims for review and produce an adjuster deployment plan.

| 5W + H | |
|---|---|
| **Who** | A carrier's catastrophe operations centre and special investigations unit (SIU). |
| **What** | A repeatable hotspot engine after a tornado plus hail event: haversine DBSCAN, data-driven eps, cluster dollar stats, k-Means comparison and a scored review queue. |
| **When** | Re-run every few hours during the first weeks after a catastrophe as claims arrive. |
| **Where** | A scheduled job on the claims platform feeding a field-deployment dashboard and SIU queue. |
| **Why** | Tornado tracks are long and narrow while hail cores are wide; the right method must follow shape, ignore isolated claims and scale to any region without retuning by hand. |
| **How** | Simulate the event, find eps from the k-distance curve knee, run DBSCAN with the haversine metric and ball tree, test eps sensitivity, compare ARI with k-Means, score noise claims with simple rules, plan adjuster-days and save the config. |

**🧠 ML concepts:** Haversine (great-circle) distance on radians, k-distance plot and knee detection for eps, Ball-tree indexing for neighbour search, Sensitivity analysis of eps, Adjusted Rand Index vs ground truth, DBSCAN vs k-Means on non-spherical clusters  
**🐍 Python concepts:** Functions returning multiple values, Tuples of specs unpacked in a for loop, Named aggregation in groupby, Boolean Series arithmetic, Category codes for labels, tempfile and joblib

In [ ]:
# Hard: catastrophe-claim hotspot engine - haversine DBSCAN, k-distance eps, k-Means comparison, noise triage
import os
import tempfile
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN, KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import adjusted_rand_score

warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)
EARTH_KM = 6371.0
ADJUSTER_CLAIMS_PER_DAY = 6


def simulate_cat_event():
    """Tornado track + two hail cores + scattered claims (some suspicious) after one storm."""
    parts = []
    # 🐍 PY: tuples of (name, lat0, lon0, lat1, lon1, n, spread_deg, mean_log_amount)
    specs = [("Tornado track", 36.05, -97.25, 36.18, -96.95, 350, 0.0025, 10.6),
             ("Hail core W", 36.25, -97.10, 36.27, -97.06, 400, 0.006, 9.2),
             ("Hail core E", 36.10, -96.85, 36.12, -96.82, 250, 0.006, 9.3)]
    for name, la0, lo0, la1, lo1, n, sd, mu in specs:
        t = rng.random(n)
        parts.append(pd.DataFrame({
            "lat": la0 + t * (la1 - la0) + rng.normal(0, sd, n),
            "lon": lo0 + t * (lo1 - lo0) + rng.normal(0, sd, n),
            "claim_usd": rng.lognormal(mu, 0.5, n), "event_zone": name}))
    n = 50
    parts.append(pd.DataFrame({"lat": rng.uniform(35.9, 36.4, n), "lon": rng.uniform(-97.4, -96.7, n),
                               "claim_usd": rng.lognormal(9.6, 0.8, n), "event_zone": "Scattered"}))
    df = pd.concat(parts, ignore_index=True)
    df["days_after_event"] = rng.integers(0, 30, len(df))
    df["policy_age_days"] = rng.integers(30, 3000, len(df))
    # A few claims on brand-new policies far from the storm - classic red flag
    df.loc[df.event_zone == "Scattered", "policy_age_days"] = rng.choice([10, 25, 45, 400, 1500], n)
    df["claim_id"] = [f"CLM{i:05d}" for i in range(len(df))]
    return df


def k_distance_eps(coords_rad, k):
    """Return the 'knee' of the sorted k-th-neighbour distance curve, in km."""
    # 🧠 ML: haversine works on (lat, lon) in RADIANS and returns great-circle angle; x Earth radius = km
    nn = NearestNeighbors(n_neighbors=k, metric="haversine").fit(coords_rad)
    d, _ = nn.kneighbors(coords_rad)
    kd = np.sort(d[:, -1]) * EARTH_KM
    # 🧠 ML: knee = point furthest from the straight line joining first and last points of the curve
    # 🐍 PY: linspace gives evenly spaced values; here the diagonal line we compare the curve to
    x = np.linspace(0, 1, len(kd))
    y = (kd - kd.min()) / (kd.max() - kd.min())
    knee = int(np.argmax(np.abs(y - x)))
    return kd, kd[knee]


def hotspot_report(df, label_col):
    g = df[df[label_col] != -1].groupby(label_col)
    # 🐍 PY: named aggregation - each keyword becomes an output column
    rep = g.agg(claims=("claim_id", "size"), total_usd=("claim_usd", "sum"),
                avg_usd=("claim_usd", "mean"), lat=("lat", "mean"), lon=("lon", "mean"))
    # 🧠 ML: turn cluster size into a resource number managers can act on
    rep["adjuster_days"] = np.ceil(rep.claims / ADJUSTER_CLAIMS_PER_DAY).astype(int)
    return rep.sort_values("total_usd", ascending=False)


df = simulate_cat_event()
# 🐍 PY: np.radians converts degrees to radians, which the haversine metric requires
coords = np.radians(df[["lat", "lon"]].to_numpy())
print(f"{len(df)} claims after the event, total incurred ${df.claim_usd.sum():,.0f}\n")

# 1. Pick eps with the k-distance plot (text version)
MIN_SAMPLES = 12
kd, eps_km = k_distance_eps(coords, MIN_SAMPLES)
print(f"k-distance curve (k={MIN_SAMPLES}) percentiles, km:")
for q in [10, 50, 80, 90, 95, 99]:
    v = np.percentile(kd, q)
    # 🐍 PY: min() caps the ASCII bar so a huge tail value does not flood the screen
    print(f"  p{q:<2} {v:5.2f} " + "#" * min(int(v * 10), 40))
print(f"Knee suggests eps = {eps_km:.2f} km\n")

# 2. DBSCAN with haversine + ball_tree
# 🧠 ML: eps must be in radians for haversine -> divide km by Earth radius
db = DBSCAN(eps=eps_km / EARTH_KM, min_samples=MIN_SAMPLES, metric="haversine", algorithm="ball_tree")
df["dbscan"] = db.fit_predict(coords)
# 🐍 PY: nunique counts distinct labels; subtract one if noise (-1) is present
n_clusters = df.dbscan.nunique() - (1 if -1 in df.dbscan.values else 0)
print(f"DBSCAN found {n_clusters} hotspots, {np.sum(df.dbscan == -1)} noise claims")
print(hotspot_report(df, "dbscan").round({"total_usd": 0, "avg_usd": 0, "lat": 3, "lon": 3}).to_string())

# 3. Sensitivity to eps
print("\n eps_km | hotspots | noise")
for f in [0.5, 1.0, 2.0]:
    lab = DBSCAN(eps=eps_km * f / EARTH_KM, min_samples=MIN_SAMPLES, metric="haversine",
                 algorithm="ball_tree").fit_predict(coords)
    print(f"  {eps_km * f:5.2f} |    {len(set(lab) - {-1}):>2}    | {np.sum(lab == -1):>4}")

# 4. Compare with k-Means (k = DBSCAN cluster count)
# 🧠 ML: k-Means must assign EVERY claim and assumes round blobs - it cannot flag outliers or follow a track
km = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
xy = np.column_stack([df.lat * 111, df.lon * 111 * np.cos(np.radians(df.lat.mean()))])
df["kmeans"] = km.fit_predict(xy)
# 🧠 ML: ARI compares two labelings and ignores label names; 1.0 = identical grouping
truth = df.event_zone.astype("category").cat.codes
print(f"\nAgreement with true storm zones (Adjusted Rand Index, 1 = perfect):")
print(f"  DBSCAN  {adjusted_rand_score(truth, df.dbscan):.3f}")
print(f"  k-Means {adjusted_rand_score(truth, df.kmeans):.3f}")
scattered_km = df[df.event_zone == "Scattered"].kmeans.value_counts().sum()
print(f"  k-Means forced all {scattered_km} scattered claims into a hotspot; "
      f"DBSCAN flagged {np.sum((df.event_zone == 'Scattered') & (df.dbscan == -1))} of them as noise")

# 5. Noise triage - a simple rule-based priority score for desk review
# 🧠 ML: noise != fraud. It is a review queue; combine with business rules for prioritisation
# 🐍 PY: .copy() avoids pandas SettingWithCopy surprises when we add columns
noise = df[df.dbscan == -1].copy()
# 🐍 PY: boolean Series add up as 0/1, giving a quick points-based score
noise["review_score"] = ((noise.policy_age_days < 60).astype(int) * 2
                         + (noise.claim_usd > df.claim_usd.quantile(0.9)).astype(int)
                         + (noise.days_after_event > 20).astype(int))
cols = ["claim_id", "lat", "lon", "claim_usd", "policy_age_days", "days_after_event", "review_score"]
print(f"\nTop noise claims for SIU/desk review (of {len(noise)}):")
print(noise.sort_values(["review_score", "claim_usd"], ascending=False)[cols].head(6)
      .round({"lat": 3, "lon": 3, "claim_usd": 0}).to_string(index=False))

# 6. Business decision: adjuster deployment
rep = hotspot_report(df, "dbscan")
print(f"\nDeploy plan: {rep.adjuster_days.sum()} adjuster-days across {len(rep)} hotspots; "
      f"largest hotspot holds {rep.total_usd.iloc[0] / df.claim_usd.sum():.0%} of incurred $")

path = os.path.join(tempfile.mkdtemp(), "cat_hotspots.joblib")
joblib.dump({"eps_km": eps_km, "min_samples": MIN_SAMPLES, "hotspots": rep}, path)
print(f"Saved hotspot config -> {os.path.basename(path)}")
print("Compliance: noise flags trigger review, not denial - state unfair claims settlement rules"
      " (NAIC model act) require prompt, fair handling of every claim.")

**Try this next:**
- Change MIN_SAMPLES to 6 and 25 and see how the knee eps and the hotspots change.
- Replace DBSCAN with sklearn's HDBSCAN (metric='haversine') and compare noise counts.
- Add a 'roof_age_years' column and include it in the review score.